<div dir="rtl" style="text-align:right;direction:rtl;font:750 19px Tahoma,Arial,sans-serif;line-height:2">
# KhaledDSML 0.8 · مختبر الوساطة والتعديل

## تحليل المسارات، Bootstrap، والتفاعل، وحدود Johnson–Neyman

تطبيق جامعي لطلبة الاقتصاد، باللغتين العربية والإنجليزية، مع واجهة Analytics Studio مريحة وجذابة، وعناوين عربية بارزة يمينًا وإنجليزية يسارًا.

**تنبيه:** الأرقام اصطناعية تعليمية، ولا تثبت نماذج الوساطة السببية. هذه نسخة تطويرية؛ النسخة الرسمية الحالية 0.7.0.
</div>

<div dir="rtl" style="text-align:right">
## 1. التثبيت في Google Colab

نفذ الخلية ثم اختر Runtime → Restart session قبل الاستيراد.
</div>

In [ ]:
%pip install "khaled-dsml[econometrics]==0.7.0"
%pip install --force-reinstall --no-deps --no-cache-dir "git+https://github.com/noble2013/KhaledDSML.git@feature/v0.8-mediation-moderation-suite"

<div dir="rtl" style="text-align:right">
## 2. المسارات الإحصائية a وb وc وc′

المتغيرات المولدة تحاكي ارتباطات خطية؛ لا تستنتج منها أن المتغير الوسيط سبّب النتيجة.
</div>

In [ ]:
import numpy as np
import khaled_dsml as kd

rng=np.random.default_rng(42)
n=160
economic_support=rng.normal(size=n)
confounder=rng.normal(size=n)
learning_effort=.65*economic_support+.2*confounder+rng.normal(size=n)
outcome=.25*economic_support+1.1*learning_effort+.2*confounder+rng.normal(size=n)

paths=kd.mediation_paths(
    economic_support,learning_effort,outcome,controls=confounder
)
sobel=kd.sobel_mediation_test(
    economic_support,learning_effort,outcome,controls=confounder
)
bootstrap=kd.bootstrap_mediation_interval(
    economic_support,learning_effort,outcome,controls=confounder,
    repetitions=500,seed=42
)
kd.display_analysis_dashboard(
    [paths,sobel,bootstrap],
    title_ar="مرصد تحليل الوساطة والمسارات",
    title_en="MEDIATION · INDIRECT ASSOCIATIONS",
    theme="aurora",font_size=22,
)

<div dir="rtl" style="text-align:right">
## 3. اختبار التفاعل والتعديل

النموذج يضيف الحد X×W فوق التأثيرات الرئيسية. تصحح HC3 الخطأ المعياري للتفاعل، أما اختبار ΔR² الكلاسيكي فيستخدم افتراض تباين ثابت.
</div>

In [ ]:
rng=np.random.default_rng(70)
x=rng.normal(size=210)
w=rng.uniform(-2,2,size=210)
y=.4*x+.8*w+.85*x*w+rng.normal(size=210)

interaction=kd.moderation_interaction_test(x,w,y,robust="HC3")
slopes=kd.simple_slopes_table(
    x,w,y,moderator_values=[-1.0,0.0,1.0],robust="HC3"
)
regions=kd.johnson_neyman_region(x,w,y,robust="HC3")

kd.display_analysis_dashboard(
    [interaction,slopes,regions],
    title_ar="لوحة اختبار التفاعل والميل المشروط",
    title_en="MODERATION · JOHNSON–NEYMAN",
    theme="amethyst",font_size=22,
)

<div dir="rtl" style="text-align:right">
## 4. تقرير جامعي متكامل أنيق

العناوين العربية داخل الجداول تظهر في الأعلى يمينًا، وأسفلها اسم العمود الإنجليزي يسارًا، مع قابلية الطباعة.
</div>

In [ ]:
saved=kd.save_analysis_dashboard(
    [paths,sobel,bootstrap,interaction,slopes,regions],
    "KhaledDSML_Mediation_Moderation_AR_EN.html",
    title_ar="تقرير تحليل الوساطة والتعديل الإحصائي",
    title_en="ACADEMIC MEDIATION AND MODERATION",
    theme="ocean",font_size=22
)
print("Saved HTML:",saved)

<div dir="rtl" style="text-align:right;line-height:1.9;font:650 16px Tahoma,Arial,sans-serif">
### تنبيهات قبل استخدام النتائج في الأطروحات

لا يثبت الدليل الإحصائي وجود تأثير سببي، وSobel تقريبي، وBootstrap يفترض استقلال وحدات العينة. ينبغي اختيار المتغيرات الضابطة وترتيبها الزمني قبل الاختبار، ومراجعة الافتراضات وتقرير أحجام الأثر والفواصل قبل النشر.
</div>